In [ ]:
#installing libraries
!pip install tensorflow==2.4.1 opencv-python mediapipe scikit-learn matplotlib

In [ ]:
# -*- coding: utf-8 -*-
"""Enhanced MSL Sign Language Recognition with LSTM, Attention, and Normalisation"""

import cv2
import numpy as np
import os
import time
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Attention
from tensorflow.keras.callbacks import TensorBoard, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical



In [ ]:
# MediaPipe setup

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))



In [ ]:
# Keypoint extraction with normalisation and without face mesh

def extract_keypoints(results):
    """
    Extracts pose (33*4) + left hand (21*3) + right hand (21*3) = 132 + 63 + 63 = 258 features
    Normalised relative to the nose (pose landmark index 0) for translation invariance.
    """
    if results.pose_landmarks is None:
        pose = np.zeros(33*4)
        lh = np.zeros(21*3)
        rh = np.zeros(21*3)
        return np.concatenate([pose, lh, rh])

    # Reference: nose (index 0)
    nose = results.pose_landmarks.landmark[0]

    # Pose: x,y,z,visibility (normalised)
    pose = []
    for lm in results.pose_landmarks.landmark:
        pose.extend([lm.x - nose.x, lm.y - nose.y, lm.z - nose.z, lm.visibility])
    pose = np.array(pose)

    # Left hand (21 landmarks, x,y,z only, no normalisation needed because relative to nose already)
    if results.left_hand_landmarks:
        lh = np.array([[lm.x - nose.x, lm.y - nose.y, lm.z - nose.z] for lm in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21*3)

    # Right hand
    if results.right_hand_landmarks:
        rh = np.array([[lm.x - nose.x, lm.y - nose.y, lm.z - nose.z] for lm in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21*3)

    return np.concatenate([pose, lh, rh])



In [ ]:
# Data collection (run once)

def collect_data(actions, no_sequences=30, sequence_length=30):
    DATA_PATH = os.path.join('MP_Data')
    for action in actions:
        for sequence in range(no_sequences):
            try:
                os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
            except:
                pass

    cap = cv2.VideoCapture(0)
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        for action in actions:
            for sequence in range(no_sequences):
                for frame_num in range(sequence_length):
                    ret, frame = cap.read()
                    image, results = mediapipe_detection(frame, holistic)
                    draw_styled_landmarks(image, results)

                    if frame_num == 0:
                        cv2.putText(image, 'STARTING COLLECTION', (120,200),
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 4, cv2.LINE_AA)
                        cv2.putText(image, f'Collecting {action} video {sequence}', (15,12),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
                        cv2.imshow('OpenCV Feed', image)
                        cv2.waitKey(2000)
                    else:
                        cv2.putText(image, f'Collecting {action} video {sequence}', (15,12),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
                        cv2.imshow('OpenCV Feed', image)

                    keypoints = extract_keypoints(results)
                    npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                    np.save(npy_path, keypoints)

                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        break
    cap.release()
    cv2.destroyAllWindows()



In [ ]:
# Loading and preparing data

def load_data(actions, no_sequences=30, sequence_length=30):
    DATA_PATH = os.path.join('MP_Data')
    sequences, labels = [], []
    for action in actions:
        for sequence in range(no_sequences):
            window = []
            for frame_num in range(sequence_length):
                res = np.load(os.path.join(DATA_PATH, action, str(sequence), f"{frame_num}.npy"))
                window.append(res)
            sequences.append(window)
            labels.append(actions.index(action))
    return np.array(sequences), np.array(labels)



In [ ]:
# LSTM model with Attention

def build_model(input_shape, num_classes):
    model = Sequential([
        LSTM(64, return_sequences=True, activation='relu', input_shape=input_shape),
        LSTM(128, return_sequences=True, activation='relu'),
        Attention(),  # Enhancenet for learning important frames
        LSTM(64, return_sequences=False, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    return model



In [ ]:
# Real-time inference with smoothing

def real_time_inference(model, actions, threshold=0.8):
    sequence = []
    pred_buffer = []   # stores recent predictions for smoothing
    sentence = []      # final recognised sentence

    cap = cv2.VideoCapture(0)
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image, results = mediapipe_detection(frame, holistic)
            draw_styled_landmarks(image, results)

            keypoints = extract_keypoints(results)
            sequence.append(keypoints)
            sequence = sequence[-30:]

            if len(sequence) == 30:
                res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
                predicted_idx = np.argmax(res)
                predicted_action = actions[predicted_idx]
                confidence = res[predicted_idx]

                # Smoothing buffer (last 5 predictions)
                pred_buffer.append(predicted_action)
                if len(pred_buffer) > 5:
                    # majority vote
                    unique, counts = np.unique(pred_buffer, return_counts=True)
                    best = unique[np.argmax(counts)]
                    if counts.max() >= 3 and confidence > threshold:
                        if len(sentence) == 0 or sentence[-1] != best:
                            sentence.append(best)
                    pred_buffer = []   # reset buffer

                # Show probability bar (optional setting)
                # image = prob_viz(res, actions, image, colors)

            # Display sentence
            cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
            cv2.putText(image, ' '.join(sentence[-5:]), (3,30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

            cv2.imshow('Sign Language Recognition', image)
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()



In [ ]:
# Main workflow

if __name__ == "__main__":
    # Define your gestures
    actions = np.array(['assalamalaikum', 'wow', 'thankx'])
    no_sequences = 30
    sequence_length = 30

    # STEP 1: Collect data (uncomment to record)
    # collect_data(actions, no_sequences, sequence_length)

    # STEP 2: Load data
    X, y = load_data(actions, no_sequences, sequence_length)

    # One-hot encode labels
    y_cat = to_categorical(y).astype(int)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.05, stratify=y, random_state=42)

    # Input shape: (30, 258) because we removed face mesh
    input_shape = (sequence_length, X.shape[2])  # X.shape[2] should be 258
    print(f"Input shape: {input_shape}")

    # Build and train model
    model = build_model(input_shape, len(actions))
    print(model.summary())

    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, verbose=1),
        EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True),
        TensorBoard(log_dir='./logs')
    ]

    history = model.fit(X_train, y_train,
                        validation_data=(X_test, y_test),
                        epochs=500,
                        callbacks=callbacks,
                        batch_size=32)

    # Evaluate
    y_pred = model.predict(X_test)
    y_true = np.argmax(y_test, axis=1)
    y_pred_labels = np.argmax(y_pred, axis=1)

    print("\n=== Classification Report ===")
    print(classification_report(y_true, y_pred_labels, target_names=actions))
    print(f"Accuracy: {accuracy_score(y_true, y_pred_labels):.4f}")
    print(f"Precision (weighted): {precision_score(y_true, y_pred_labels, average='weighted'):.4f}")
    print(f"Recall (weighted): {recall_score(y_true, y_pred_labels, average='weighted'):.4f}")

    # Save model
    model.save('sign_language_lstm_attention.h5')

    # STEP 3: Real-time recognition
    real_time_inference(model, actions, threshold=0.8)